In [29]:
!ls "sample_data"

anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [30]:
!python -m pip install onnx

In [31]:
import onnx
from onnx import helper
from onnx import TensorProto

import os
from typing import Optional

from google.colab import drive

In [32]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [33]:
def ensure_extension(model_name: str, extension: str = ".onnx") -> str:
    if not model_name:
        raise ValueError("Model name cannot be empty")
    if not model_name.endswith(extension):
        return f"{model_name}{extension}"
    return model_name

In [34]:
def get_onnx_path(file_path: str, model_name: str, base_dir: Optional[str] = None) -> str:
    if not file_path:
        raise ValueError("Invalid or empty file path")

    model_name = ensure_extension(model_name)

    # Use /content/drive/MyDrive as the base if file_path points to Google Drive
    if file_path.startswith('/content/drive'):
        base_path = '/content/drive/MyDrive'
    else:
        base_path = os.path.dirname(os.path.abspath(file_path))

    # Set model_dir
    if base_dir:
        model_dir = base_dir if os.path.isabs(base_dir) else os.path.join(base_path, base_dir)
    else:
        model_dir = os.path.join(base_path, "models")

    try:
        os.makedirs(model_dir, exist_ok=True)
    except OSError as e:
        raise OSError(f"Failed to create directory {model_dir}: {e}")

    return os.path.normpath(os.path.join(model_dir, model_name))

In [35]:
def create_onnx() -> onnx.ModelProto:
    try:
        # Create input and output tensors
        a = helper.make_tensor_value_info('a', TensorProto.FLOAT, [10, 10])
        x = helper.make_tensor_value_info('x', TensorProto.FLOAT, [10, 10])
        b = helper.make_tensor_value_info('b', TensorProto.FLOAT, [10, 10])
        y = helper.make_tensor_value_info('y', TensorProto.FLOAT, [10, 10])

        # Create computation nodes
        mul = helper.make_node('Mul', ['a', 'x'], ['c'], name='multiply')
        add = helper.make_node('Add', ['c', 'b'], ['y'], name='add')

        # Create graph
        graph = helper.make_graph([mul, add], 'sample-linear', [a, x, b], [y])

        # Create model
        model = helper.make_model(
            graph,
            producer_name='onnx_example',
            opset_imports=[helper.make_operatorsetid('', 15)]
        )
        model.ir_version = 8

        # Validate model
        onnx.checker.check_model(model)
        print("Model is valid. Model information:")
        print(onnx.helper.printable_graph(model.graph))

        # Save to Google Drive under /content/drive/MyDrive/models
        output_path = get_onnx_path('/content/drive/MyDrive', "sample-linear.onnx", base_dir="models")
        onnx.save(model, output_path)
        print(f"Model saved to: {output_path}")

        # Verify file exists
        if os.path.exists(output_path):
            print("File successfully saved!")
        else:
            print("File not found after saving!")

        return model

    except Exception as e:
        print(f"Error in create_onnx: {e}")
        raise

# Run the function
model = create_onnx()

Model is valid. Model information:
graph sample-linear (
  %a[FLOAT, 10x10]
  %x[FLOAT, 10x10]
  %b[FLOAT, 10x10]
) {
  %c = Mul(%a, %x)
  %y = Add(%c, %b)
  return %y
}
Model saved to: /content/drive/MyDrive/models/sample-linear.onnx
File successfully saved!
